# Camada Gold Financeira — Terra Signal

Nesta etapa, serão preparados dados financeiros a partir da camada Silver, com o objetivo de analisar a receita associada aos clientes, os impactos dos cancelamentos e os segmentos que merecem maior atenção.

O objetivo é organizar informações financeiras para a construção de indicadores e análises que auxiliem na identificação de possíveis perdas de receita e oportunidades de retenção de clientes.

### Considerações sobre os dados

A base utilizada não apresenta as datas dos cancelamentos. Portanto, os valores mensais associados aos clientes que cancelaram serão tratados como estimativas de receita mensal potencialmente perdida, e não como perdas financeiras efetivamente registradas em determinado mês.

Os registros com valores acumulados ausentes serão considerados apenas nas análises em que existam informações suficientes, evitando a criação de valores sem fundamento.

In [2]:
import pandas as pd
from pathlib import Path

pasta_silver = Path("dados/silver")

df_financeiro = pd.read_csv(pasta_silver / "history_silver.csv")

df_financeiro.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome,TotalCharges_Estimado,tenure_numerico
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,...,One year,Yes,Electronic check,68.75,3482.85,No,I have been a customer with this internet prov...,6532,3482.85,52.0
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,...,One year,Yes,Mailed check,73.90,2405.05,Yes,I've been a customer with this company for ove...,7634,2405.05,33.0
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,...,One year,Yes,Mailed check,20.05,417.00,No,I have been a customer with this company for 2...,3628,417.00,22.0
3,5697-GOMBF,Female,1,Yes,Yes,28,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,35.90,973.65,No,I have been a customer with this internet prov...,7851,973.65,28.0
4,9717-QEBGU,Male,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.30,144.00,No,I have been using the fiber optic internet ser...,1691,144.00,2.0


- import pandas as pd: importa a biblioteca Pandas.
- from pathlib import Path: permite trabalhar com caminhos de arquivos.
- pasta_silver: indica onde está nosso arquivo tratado.
- pd.read_csv(): carrega os dados da Silver para um DataFrame chamado df_financeiro.
- df_financeiro.head(): apresenta os cinco primeiros registros para conferência.

Por que df_financeiro?

Porque vamos utilizar esse DataFrame como ponto de partida para preparar nossas tabelas financeiras, sem modificar o arquivo original da Silver.

### Seleção das variáveis financeiras

Nesta etapa, serão selecionadas as colunas da camada Silver necessárias para as análises financeiras.

Serão utilizadas informações sobre identificação dos clientes, tipos de contrato, serviços de internet, cancelamentos, valores mensais, valores acumulados e tempo de permanência.

A seleção permite trabalhar apenas com os dados relevantes para os indicadores financeiros, mantendo a base original da camada Silver preservada.

In [3]:
df_financeiro = df_financeiro[[
    "customerID",
    "Contract",
    "InternetService",
    "Churn",
    "MonthlyCharges",
    "TotalCharges_Estimado",
    "tenure_numerico"
]].copy()

df_financeiro.head()

,customerID,Contract,InternetService,Churn,MonthlyCharges,TotalCharges_Estimado,tenure_numerico
0,4578-PHJYZ,One year,DSL,No,68.75,3482.85,52.0
1,6289-CPNLD,One year,DSL,Yes,73.90,2405.05,33.0
2,2682-KEVRP,One year,No,No,20.05,417.00,22.0
3,5697-GOMBF,Month-to-month,DSL,No,35.90,973.65,28.0
4,9717-QEBGU,Month-to-month,Fiber optic,No,70.30,144.00,2.0


### Análise da receita mensal por situação do cliente

Nesta etapa, será analisada a receita mensal associada aos clientes que permaneceram e aos que cancelaram os serviços.

A análise utiliza os valores da coluna `MonthlyCharges`, agrupados conforme a situação de cancelamento (`Churn`).

O objetivo é identificar o valor mensal associado a cada grupo e estimar a receita mensal potencialmente perdida com os cancelamentos.

Como a base não apresenta as datas dos cancelamentos, os valores dos clientes que cancelaram serão considerados estimativas de receita mensal potencialmente perdida, e não perdas financeiras efetivamente registradas em um determinado mês.

In [4]:
tabela_receita = df_financeiro.groupby("Churn")["MonthlyCharges"].sum().round(2)

tabela_receita

Churn
No     251767.15
Yes    112242.90
Name: MonthlyCharges, dtype: float64

### Resultado da análise da receita mensal

Os clientes que permaneceram apresentaram um valor mensal acumulado de R$ 251.767,15, enquanto os clientes que cancelaram representavam R$ 112.242,90 em mensalidades.

Somando os dois grupos, obtemos um valor mensal total de R$ 364.010,05.

Os clientes que cancelaram representavam aproximadamente 30,84% desse valor total.

Como não temos as datas dos cancelamentos, os R$ 112.242,90 representam uma estimativa de receita mensal potencialmente perdida, e não uma perda efetivamente registrada em determinado mês.

### Indicadores de receita mensal

Nesta etapa, serão calculados indicadores financeiros a partir dos valores mensais associados aos clientes que permaneceram e aos que cancelaram os serviços.

Serão apresentados o valor mensal total da carteira, a receita mensal associada aos clientes ativos, a receita mensal potencialmente perdida e o percentual que essa perda representa em relação ao total.

Esses indicadores permitirão avaliar a dimensão financeira dos cancelamentos e apoiar as próximas análises.

In [5]:

receita_ativa = tabela_receita["No"]
receita_perdida = tabela_receita["Yes"]

receita_total = receita_ativa + receita_perdida

percentual_perda = (receita_perdida / receita_total * 100).round(2)

print(f"Valor mensal total da carteira: R$ {receita_total:.2f}")
print(f"Receita mensal dos clientes ativos: R$ {receita_ativa:.2f}")
print(f"Receita mensal potencialmente perdida: R$ {receita_perdida:.2f}")
print(f"Percentual de receita potencialmente perdida: {percentual_perda:.2f}%")


Valor mensal total da carteira: R$ 364010.05
Receita mensal dos clientes ativos: R$ 251767.15
Receita mensal potencialmente perdida: R$ 112242.90
Percentual de receita potencialmente perdida: 30.84%


### Interpretação dos indicadores financeiros

O valor mensal total associado à carteira analisada é de R$ 364.010,05, sendo R$ 251.767,15 referentes aos clientes que permaneceram e R$ 112.242,90 aos clientes que cancelaram os serviços.

Os valores mensais associados aos clientes que cancelaram representam aproximadamente 30,84% do total da carteira analisada.

Esse resultado demonstra a relevância financeira dos cancelamentos e indica a importância de identificar quais grupos de clientes concentram os maiores valores de receita potencialmente perdida.

Vale destacar que os valores representam uma estimativa baseada nas mensalidades dos clientes, não sendo possível determinar as perdas financeiras efetivamente registradas em um período específico.

### Receita mensal potencialmente perdida por tipo de contrato

Nesta etapa, serão analisados os valores mensais associados aos clientes que cancelaram os serviços, considerando os diferentes tipos de contrato.

O objetivo é identificar quais modalidades contratuais concentram os maiores valores de receita potencialmente perdida e quais merecem maior atenção nas estratégias de retenção de clientes.

In [6]:
df_cancelados = df_financeiro[df_financeiro["Churn"] == "Yes"].copy()

receita_por_contrato = df_cancelados.groupby("Contract")["MonthlyCharges"].sum().round(2)

receita_por_contrato

Contract
Month-to-month    96852.45
One year          11598.10
Two year           3792.35
Name: MonthlyCharges, dtype: float64

df_cancelados: cria um DataFrame contendo apenas os clientes que cancelaram.

groupby("Contract"): agrupa os clientes por tipo de contrato.

sum(): soma as mensalidades associadas aos cancelamentos de cada grupo.

round(2): mantém duas casas decimais.

### Participação percentual por tipo de contrato

Além dos valores mensais associados aos cancelamentos, será calculada a participação percentual de cada tipo de contrato sobre o valor total potencialmente perdido.

Essa informação permitirá identificar quais modalidades contratuais concentram o maior impacto financeiro.

In [7]:
tabela_contratos_financeiro = receita_por_contrato.reset_index()

tabela_contratos_financeiro.columns = [
    "Tipo de contrato",
    "Receita mensal potencialmente perdida"
]

tabela_contratos_financeiro["Participação (%)"] = (
    tabela_contratos_financeiro["Receita mensal potencialmente perdida"]
    / receita_perdida * 100
).round(2)

tabela_contratos_financeiro

,Tipo de contrato,Receita mensal potencialmente perdida,Participação (%)
0,Month-to-month,96852.45,86.29
1,One year,11598.10,10.33
2,Two year,3792.35,3.38


reset_index(): transforma o resultado anterior em um DataFrame.

columns: renomeia as colunas para facilitar a leitura.

Participação (%): calcula quanto cada tipo de contrato representa sobre os R$ 112.242,90 potencialmente perdidos.

round(2): mantém duas casas decimais.

### Interpretação dos resultados

Os contratos mensais concentram o maior valor de receita mensal potencialmente perdida, totalizando R$ 96.852,45, o equivalente a 86,29% do total associado aos cancelamentos.

Os contratos anuais representam R$ 11.598,10 (10,33%), enquanto os contratos bienais representam R$ 3.792,35 (3,38%).

Ao relacionar esses resultados com a análise exploratória realizada anteriormente, observa-se que os contratos mensais, além de apresentarem a maior taxa de cancelamento, também concentram o maior impacto financeiro potencial.

Esses resultados sugerem que os clientes com contratos mensais devem receber atenção prioritária nas estratégias de retenção da empresa.

### Receita mensal potencialmente perdida por serviço de internet

Nesta etapa, serão analisados os valores mensais associados aos clientes que cancelaram os serviços, considerando os diferentes tipos de serviço de internet.

O objetivo é identificar quais serviços concentram os maiores valores de receita potencialmente perdida e relacionar esses resultados com as taxas de cancelamento identificadas na análise exploratória.

In [8]:
receita_por_internet = df_cancelados.groupby("InternetService")["MonthlyCharges"].sum().round(2)

receita_por_internet

InternetService
DSL            18261.0
Fiber optic    92168.9
No              1813.0
Name: MonthlyCharges, dtype: float64

### Participação percentual por serviço de internet

Será calculada a participação percentual de cada serviço de internet sobre o valor total de receita mensal potencialmente perdida.

Essa informação permitirá identificar quais serviços concentram o maior impacto financeiro associado aos cancelamentos.

In [9]:
tabela_internet_financeiro = receita_por_internet.reset_index()

tabela_internet_financeiro.columns = [
    "Serviço de internet",
    "Receita mensal potencialmente perdida"
]

tabela_internet_financeiro["Participação (%)"] = (
    tabela_internet_financeiro["Receita mensal potencialmente perdida"]
    / receita_perdida * 100
).round(2)

tabela_internet_financeiro

,Serviço de internet,Receita mensal potencialmente perdida,Participação (%)
0,DSL,18261.0,16.27
1,Fiber optic,92168.9,82.12
2,No,1813.0,1.62


### Interpretação dos resultados

Os clientes com serviço de fibra óptica concentram o maior valor de receita mensal potencialmente perdida, totalizando R$ 92.168,90, o equivalente a 82,12% do total.

Os clientes com serviço DSL representam R$ 18.261,00 (16,27%), enquanto os clientes sem serviço de internet representam R$ 1.813,00 (1,62%).

Ao relacionar esses resultados com a análise exploratória, observa-se que a fibra óptica apresenta tanto a maior taxa de cancelamento quanto a maior concentração de receita potencialmente perdida.

Esses resultados indicam que os clientes de fibra óptica devem receber atenção especial na definição de estratégias de retenção, considerando a relevância financeira desse segmento.

### Receita mensal potencialmente perdida por tempo de permanência

Nesta etapa, será analisada a receita mensal potencialmente perdida conforme o tempo de permanência dos clientes que cancelaram os serviços.

Serão utilizadas as mesmas faixas de permanência definidas na análise exploratória, permitindo relacionar as taxas de cancelamento com seus respectivos impactos financeiros.

In [10]:
df_cancelados = df_cancelados.copy()

df_cancelados["faixa_permanencia"] = pd.cut(
    df_cancelados["tenure_numerico"],
    bins=[0, 12, 24, 48, float("inf")],
    labels=[
        "Até 12 meses",
        "13 a 24 meses",
        "25 a 48 meses",
        "Acima de 48 meses"
    ],
    include_lowest=True
)

df_cancelados[["tenure_numerico", "faixa_permanencia"]].head()

,tenure_numerico,faixa_permanencia
1,33.0,25 a 48 meses
5,70.0,Acima de 48 meses
7,15.0,13 a 24 meses
12,1.0,Até 12 meses
16,67.0,Acima de 48 meses


### Verificação dos registros sem tempo de permanência

Antes do agrupamento financeiro por faixa de permanência, serão verificados os clientes cancelados que não possuem informações válidas sobre o tempo de relacionamento com a empresa.

Essa verificação permitirá identificar o valor mensal associado aos registros que não puderam ser classificados.

In [11]:
df_sem_permanencia = df_cancelados[
    df_cancelados["faixa_permanencia"].isna()
]

print(f"Clientes sem faixa de permanência: {len(df_sem_permanencia)}")

print(f"Valor mensal associado: R$ {df_sem_permanencia['MonthlyCharges'].sum():.2f}")

Clientes sem faixa de permanência: 8
Valor mensal associado: R$ 701.80


### Resultado da verificação

Foram identificados 8 clientes cancelados sem informações válidas sobre o tempo de permanência, representando um valor mensal de R$ 701,80.

Esses registros serão classificados na categoria "Não informado", permitindo preservar os valores financeiros na análise sem atribuir uma faixa de permanência incorreta.

In [12]:
df_cancelados["faixa_permanencia"] = (
    df_cancelados["faixa_permanencia"]
    .cat.add_categories(["Não informado"])
    .fillna("Não informado")
)

df_cancelados["faixa_permanencia"].value_counts(sort=False)

faixa_permanencia
Até 12 meses         819
13 a 24 meses        229
25 a 48 meses        280
Acima de 48 meses    168
Não informado          8
Name: count, dtype: int64

cat.add_categories(): acrescenta uma nova categoria às faixas existentes.

fillna(): substitui os valores ausentes por "Não informado".

value_counts(sort=False): contabiliza os clientes por faixa, mantendo a ordem das categorias.

### Participação financeira por tempo de permanência

Será calculada a receita mensal potencialmente perdida por faixa de permanência, juntamente com a participação percentual de cada grupo em relação ao valor total associado aos cancelamentos.

Os clientes sem tempo de permanência informado também serão considerados, garantindo que todos os valores financeiros sejam contemplados na análise.

In [13]:
receita_por_permanencia = df_cancelados.groupby(
    "faixa_permanencia", observed=True
)["MonthlyCharges"].sum().round(2)

tabela_permanencia_financeiro = receita_por_permanencia.reset_index()

tabela_permanencia_financeiro.columns = [
    "Tempo de permanência",
    "Receita mensal potencialmente perdida"
]

tabela_permanencia_financeiro["Participação (%)"] = (
    tabela_permanencia_financeiro["Receita mensal potencialmente perdida"]
    / receita_perdida * 100
).round(2)

tabela_permanencia_financeiro

,Tempo de permanência,Receita mensal potencialmente perdida,Participação (%)
0,Até 12 meses,54650.60,48.69
1,13 a 24 meses,17967.15,16.01
2,25 a 48 meses,23473.00,20.91
3,Acima de 48 meses,15450.35,13.77
4,Não informado,701.80,0.63


### Interpretação dos resultados

Os clientes com até 12 meses de permanência apresentaram o maior valor de receita mensal potencialmente perdida, totalizando R$ 54.650,60, o equivalente a 48,69% do valor associado aos cancelamentos.

Em seguida, destacam-se os clientes com 25 a 48 meses de permanência (20,91%), de 13 a 24 meses (16,01%) e acima de 48 meses (13,77%).

Os clientes sem tempo de permanência informado representam apenas 0,63% do valor total.

Relacionando esses resultados com a análise exploratória, observa-se que os clientes com até 12 meses de permanência apresentam tanto a maior taxa de cancelamento quanto o maior impacto financeiro associado aos cancelamentos.

Esses resultados reforçam a importância de direcionar estratégias de retenção principalmente aos clientes durante o primeiro ano de relacionamento com a empresa.

## Análise financeira cruzada por contrato e serviço de internet

Nesta etapa, serão cruzadas as informações sobre os tipos de contrato e os serviços de internet dos clientes que cancelaram.

O objetivo é identificar quais combinações concentram os maiores valores de receita mensal potencialmente perdida, complementando as análises individuais realizadas anteriormente.

Essa análise permitirá direcionar estratégias de retenção para grupos mais específicos de clientes.

In [18]:
receita_contrato_internet = df_cancelados.groupby(
    ["Contract", "InternetService"]
)["MonthlyCharges"].sum().round(2)

tabela_contrato_internet = receita_contrato_internet.reset_index()

tabela_contrato_internet.columns = [
    "Tipo de contrato",
    "Serviço de internet",
    "Receita mensal potencialmente perdida"
]

tabela_contrato_internet["Participação (%)"] = (
    tabela_contrato_internet["Receita mensal potencialmente perdida"]
    / receita_perdida * 100
).round(2)

tabela_contrato_internet

,Tipo de contrato,Serviço de internet,Receita mensal potencialmente perdida,Participação (%)
0,Month-to-month,DSL,14850.70,13.23
1,Month-to-month,Fiber optic,80492.50,71.71
2,Month-to-month,No,1509.25,1.34
3,One year,DSL,2679.40,2.39
4,One year,Fiber optic,8728.45,7.78
5,One year,No,190.25,0.17
6,Two year,DSL,730.90,0.65
7,Two year,Fiber optic,2947.95,2.63
8,Two year,No,113.50,0.10


### Interpretação dos resultados

Os clientes com contrato mensal e serviço de fibra óptica concentram o maior impacto financeiro associado aos cancelamentos, totalizando R$ 80.492,50, equivalente a 71,71% da receita mensal potencialmente perdida.

Em seguida, destacam-se os clientes com contrato mensal e serviço DSL, representando 13,23%, e os clientes com contrato anual e fibra óptica, representando 7,78%.

O cruzamento das informações permite identificar de forma mais específica os grupos que concentram os maiores impactos financeiros associados aos cancelamentos.

Esses resultados indicam que os clientes com contrato mensal e fibra óptica devem receber atenção especial nas estratégias de retenção.

## Validação dos resultados financeiros

Nesta etapa, serão conferidos os valores totais de receita mensal potencialmente perdida nos agrupamentos por tipo de contrato, serviço de internet, tempo de permanência e no cruzamento entre contrato e serviço de internet.

O objetivo é garantir a consistência dos resultados antes da exportação dos arquivos da camada Gold Financeira.

In [19]:
print(f"Receita potencialmente perdida: R$ {receita_perdida:.2f}")
print(f"Total por contrato: R$ {receita_por_contrato.sum():.2f}")
print(f"Total por internet: R$ {receita_por_internet.sum():.2f}")
print(f"Total por permanência: R$ {receita_por_permanencia.sum():.2f}")
print(f"Total por contrato e internet: R$ {receita_contrato_internet.sum():.2f}")

Receita potencialmente perdida: R$ 112242.90
Total por contrato: R$ 112242.90
Total por internet: R$ 112242.90
Total por permanência: R$ 112242.90
Total por contrato e internet: R$ 112242.90


### Resultado da validação

Os agrupamentos por tipo de contrato, serviço de internet, tempo de permanência e o cruzamento entre contrato e serviço de internet apresentaram o mesmo valor total de receita mensal potencialmente perdida (R$ 112.242,90).

A conferência confirma a consistência dos valores financeiros entre os agrupamentos, permitindo prosseguir com a exportação das tabelas da camada Gold Financeira.

### Preparação da tabela de indicadores financeiros

Nesta etapa, os indicadores financeiros calculados anteriormente serão organizados em uma tabela para posterior exportação.

A tabela permitirá utilizar esses indicadores no notebook de análise financeira sem a necessidade de realizar novamente os cálculos.

In [15]:
tabela_indicadores_financeiros = pd.DataFrame({
    "Indicador": [
        "Valor mensal total da carteira",
        "Receita mensal dos clientes ativos",
        "Receita mensal potencialmente perdida",
        "Percentual de receita potencialmente perdida"
    ],
    "Valor": [
        receita_total,
        receita_ativa,
        receita_perdida,
        percentual_perda
    ],
    "Unidade": [
        "R$",
        "R$",
        "R$",
        "%"
    ]
})

tabela_indicadores_financeiros

,Indicador,Valor,Unidade
0,Valor mensal total da carteira,364010.05,R$
1,Receita mensal dos clientes ativos,251767.15,R$
2,Receita mensal potencialmente perdida,112242.90,R$
3,Percentual de receita potencialmente perdida,30.84,%


### Exportação das tabelas financeiras

Após a preparação e validação dos resultados financeiros, as tabelas serão exportadas em formato CSV para a pasta `dados/gold/`.

Os arquivos serão utilizados posteriormente no notebook de análise financeira para a construção dos gráficos e interpretação dos resultados.

In [20]:
from pathlib import Path

pasta_gold = Path("dados/gold")

pasta_gold.mkdir(parents=True, exist_ok=True)

tabela_indicadores_financeiros.to_csv(
    pasta_gold / "indicadores_financeiros.csv",
    index=False
)

tabela_contratos_financeiro.to_csv(
    pasta_gold / "contratos_financeiro.csv",
    index=False
)

tabela_internet_financeiro.to_csv(
    pasta_gold / "internet_financeiro.csv",
    index=False
)

tabela_permanencia_financeiro.to_csv(
    pasta_gold / "permanencia_financeiro.csv",
    index=False
)

tabela_contrato_internet.to_csv(
    pasta_gold / "contrato_internet_financeiro.csv",
    index=False
)

In [21]:
list(pasta_gold.iterdir())

[WindowsPath('dados/gold/.gitkeep'),
 WindowsPath('dados/gold/contratos.csv'),
 WindowsPath('dados/gold/contratos_financeiro.csv'),
 WindowsPath('dados/gold/contrato_internet_financeiro.csv'),
 WindowsPath('dados/gold/indicadores_financeiros.csv'),
 WindowsPath('dados/gold/internet.csv'),
 WindowsPath('dados/gold/internet_financeiro.csv'),
 WindowsPath('dados/gold/permanencia.csv'),
 WindowsPath('dados/gold/permanencia_financeiro.csv')]

## Finalização da camada Gold Financeira

Após as etapas de preparação, agrupamento e validação dos dados financeiros, foram gerados cinco arquivos na pasta `dados/gold`:

- `indicadores_financeiros.csv`
- `contratos_financeiro.csv`
- `internet_financeiro.csv`
- `permanencia_financeiro.csv`
- `contrato_internet_financeiro.csv`

Os arquivos apresentam os principais indicadores financeiros e os valores de receita mensal potencialmente perdida, organizados por diferentes segmentos de clientes.

Os resultados foram validados e estão disponíveis para utilização no notebook de análise financeira.